# درس ۰۵ - RAG عاملیت‌گرا


## راه‌اندازی

این دفترچه یادداشت الگوی Agentic RAG (تولید با افزایش بازیابی) را با استفاده از چارچوب عامل مایکروسافت نشان می‌دهد.

**پیش‌نیازها:**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — نقطه پایانی سرویس جستجوی Azure AI شما
- `AZURE_SEARCH_API_KEY` — کلید API سرویس جستجوی Azure AI شما
- استقرار Azure OpenAI که از طریق متغیرهای محیطی پیکربندی شده است
- ورود به سیستم Azure CLI (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## RAG عاملی چیست؟

RAG سنتی یک روند ثابت را دنبال می‌کند: ابتدا اسناد را بازیابی می‌کند، سپس پاسخ می‌دهد. **RAG عاملی** فراتر می‌رود و به عامل استقلال می‌دهد تا تصمیم بگیرد **چه زمانی** و **چگونه** اطلاعات را بازیابی کند.

با RAG عاملی، عامل می‌تواند:
- **تصمیم بگیرد** که آیا قبل از پاسخ دادن به سوال بازیابی لازم است یا خیر
- **انتخاب کند** کدام منبع داده یا ابزار را استعلام کند
- **ارزیابی کند** نتایج بازیابی شده را و در صورت ناکافی بودن تلاش اول، بازیابی‌های بعدی انجام دهد
- **ترکیب کند** اطلاعات حاصل از چندین مرحله بازیابی را در یک پاسخ منسجم

این کار عامل را نسبت به یک روند ثابت بازیابی-سپس-تولید پاسخ، انعطاف‌پذیرتر و دقیق‌تر می‌کند.


## ایجاد ابزار جستجو

در Agentic RAG، منابع داده خارجی به‌عنوان **ابزارها** بسته‌بندی می‌شوند که عامل می‌تواند در صورت نیاز فراخوانی کند. این امکان را به عامل می‌دهد که بازیابی را صرفاً به‌عنوان یک عمل دیگر که می‌تواند انجام دهد در نظر بگیرد، نه یک مرحله اجباری.

در ادامه، یک پایگاه دانش سفر تعریف کرده و آن را به‌صورت ابزاری که عامل می‌تواند برای جستجوی اطلاعات مقصد فراخوانی کند، در دسترس قرار می‌دهیم.


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {
    "Barcelona": "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.",
    "Tokyo": "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.",
    "Paris": "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.",
    "Cape Town": "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150.",
}


@tool(approval_mode="never_require")
def search_travel_knowledge(
    query: Annotated[str, "The search query about a travel destination"]
) -> str:
    """Search the travel knowledge base for destination information."""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any(
            word in info.lower() for word in query.lower().split()
        ):
            results.append(f"**{destination}**: {info}")
    return (
        "\n\n".join(results)
        if results
        else "No matching destinations found in the knowledge base."
    )

## ساخت عامل RAG

اکنون عاملی می‌سازیم که دستور دارد **همیشه قبل از پاسخگویی اطلاعات را بازیابی کند**. عامل از ابزار `search_travel_knowledge` استفاده می‌کند تا پاسخ‌های خود را بر اساس پایگاه دانش بسازد و به داده‌های آموزشی خود متکی نباشد.


In [ ]:
agent = client.as_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGAgent",
    instructions="""You are a knowledgeable travel advisor. Before answering questions about destinations:
1. ALWAYS search the travel knowledge base first
2. Base your answers on retrieved information
3. If information is not in the knowledge base, say so clearly
4. Provide specific details like costs, best seasons, and highlights.""",
)

response = await agent.run(
    "I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    )
print(response)

## بازیابی تکراری — الگوی سازنده-بازرس

یک مزیت کلیدی Agentic RAG، **بازیابی تکراری** است. عامل می‌تواند چندین دور جستجو انجام دهد تا یافته‌های اولیه خود را تأیید، اصلاح یا گسترش دهد — مشابه جریان کاری "سازنده-بازرس":

1. **مرحله سازنده**: عامل اطلاعات اولیه را بازیابی کرده و پاسخ پیش‌نویس می‌کند.
2. **مرحله بازرس**: عامل بازیابی‌های اضافی انجام می‌دهد تا جزئیات را تأیید کند یا خلاها را پر کند.

در زیر، از عامل سوالی پرسیده می‌شود که نیاز به مقایسه چندین مقصد دارد، که او را وادار به جستجو چندباره می‌کند.


In [ ]:
checker_agent = client.as_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGCheckerAgent",
    instructions="""You are a meticulous travel advisor who double-checks recommendations.
When answering travel questions:
1. Search for relevant destinations first
2. For each destination found, search again with the destination name to get full details
3. Compare the options using verified information
4. Present a final recommendation with specific costs, best travel times, and highlights
5. If any detail seems incomplete, search once more to confirm before responding.""",
)

response = await checker_agent.run(
    "I have a $175/day budget and want to travel in April. Which destinations fit my budget and timing?",
    )
print(response)

## خلاصه

در این درس یاد گرفتید چگونه یک سیستم **Agentic RAG** با استفاده از Microsoft Agent Framework بسازید:

- **Agentic RAG** به عوامل اجازه می‌دهد به‌طور خودمختار تصمیم بگیرند چه زمانی اطلاعات را بازیابی کنند، و این بازیابی را به صورت پویا به جای ثابت انجام می‌دهد.
- **ابزارها به عنوان منابع داده**: پایگاه‌های دانش خارجی (مانند Azure AI Search) به عنوان ابزارهایی بسته‌بندی شده‌اند که عامل می‌تواند آن‌ها را فراخوانی کند.
- **بازیابی تکراری**: الگوی maker-checker به عامل امکان می‌دهد چندین دور بازیابی انجام دهد — جستجو، تأیید و پالایش — پیش از ارائه پاسخ نهایی.

در محیط تولید، به جای `TRAVEL_KNOWLEDGE_BASE` در حافظه، از یک شاخص جستجوی واقعی Azure AI برای مدیریت بازیابی اسناد سفر در مقیاس بزرگ استفاده خواهید کرد.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**سلب مسئولیت**:
این سند با استفاده از سرویس ترجمه هوش مصنوعی [Co-op Translator](https://github.com/Azure/co-op-translator) ترجمه شده است. در حالی که ما در تلاش برای دقت هستیم، لطفاً توجه داشته باشید که ترجمه‌های خودکار ممکن است شامل خطاها یا نادرستی‌هایی باشند. سند اصلی به زبان مادری خود باید به عنوان منبع معتبر در نظر گرفته شود. برای اطلاعات حیاتی، ترجمه حرفه‌ای انسانی توصیه می‌شود. ما در قبال هرگونه سوء تفاهم یا برداشت نادرست ناشی از استفاده از این ترجمه مسئولیتی نداریم.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
